# 11 — Graph vs kNN communities (ARI)

Louvain on **PPI G** vs Louvain on **mutual kNN** graph from embedding features. Requires **NetworkX ≥ 2.8** (`louvain_communities`).


### Louvain on PPI vs mutual kNN graph (ARI)

**Method:** Same vertex subset (capped). **Louvain** on the induced PPI subgraph `G`, and Louvain on a **mutual kNN** graph: two vertices are adjacent only if each is among the other’s k nearest neighbors in **[direction | log1p(Inf.Hyp.Rad)]** Euclidean space. **ARI** compares the two labelings.

**How to read the output:** **ARI = 1** means identical partitions (up to label permutation); **0** is chance-level for large balanced label counts; **negative** is worse than random. Moderate positive ARI suggests some alignment between topology- and embedding-defined communities; low ARI means the kNN geometry disagrees with PPI modularity—both are plausible depending on the model.


In [ ]:
from pathlib import Path
import numpy as np
import networkx as nx
from sklearn.metrics import adjusted_rand_score
from sklearn.neighbors import NearestNeighbors

import dmercator3d_io as dm

RUN_SUBDIR = "d3"
paths = dm.paths_for_run(RUN_SUBDIR)
_, df = dm.parse_inf_coord(paths["inf_coord"])
G = dm.load_edges_graph(paths["edge"])
rng = np.random.default_rng(7)
vert_set = set(df["Vertex"].astype(str))
nodes = [n for n in G.nodes() if str(n) in vert_set]
if len(nodes) > 5000:
    pick = rng.choice(len(nodes), size=5000, replace=False)
    nodes = [nodes[i] for i in pick]
subG = G.subgraph(nodes).copy()
name_to_i = {str(v): i for i, v in enumerate(df["Vertex"])}
row_idx = np.array([name_to_i[str(n)] for n in nodes], dtype=int)
U = dm.normalize_direction_nd(df)
X = U[row_idx]
X = np.hstack([X, np.log1p(df["Inf.Hyp.Rad"].to_numpy()[row_idx, None])])

k = 15
nn = NearestNeighbors(n_neighbors=k + 1).fit(X)
_, ind = nn.kneighbors(X)
nloc = len(nodes)
nbr_sets = [set(ind[i, 1:].tolist()) for i in range(nloc)]
Kn = nx.Graph()
Kn.add_nodes_from(nodes)
for i in range(nloc):
    for j in nbr_sets[i]:
        j = int(j)
        if i in nbr_sets[j]:
            Kn.add_edge(nodes[i], nodes[j])

part_g = nx.community.louvain_communities(subG, seed=7)
part_k = nx.community.louvain_communities(Kn, seed=7)


def comm_to_labels(comms, nodelist):
    lab = {}
    for i, c in enumerate(comms):
        for v in c:
            lab[v] = i
    return [lab[v] for v in nodelist]


lg = comm_to_labels(part_g, nodes)
lk = comm_to_labels(part_k, nodes)
print("ARI(Louvain_G, Louvain_kNN_mutual):", adjusted_rand_score(lg, lk))
